# Problem framing & setup

**Input:**
* lyrics only (text)

**Output:**

* Single best Raag (argmax of probabilities)

* Optionally: top-k Raags with probabilities.

**Raag set:** full set vs. a smaller subset (e.g., 5–10 Raags) to start.

**Goal:** Given devotional lyrics (a shabad/bandish), predict which Raag it belongs to (or which Raag is the best fit).
* Type of problem: Multi-class text classification (Raag is the class label).

## Step 1: Open the PDF and read only the 1st page that has Raag labels

In [1]:
import pdfplumber

def get_page_text(pdf_path, page_number=0):
    # open the pdf
    try:
         with pdfplumber.open(pdf_path) as pdf:
            page = pdf.pages[page_number]
            # Extract text from that page
            text = page.extract_text()
            if text:
                print(f'Text extracted from page {page_number + 1}!')
            else:
                print(f"No text found on page {page_number + 1}.")
                return None
            return text
    except FileNotFoundError:
        print(f'Cannot find the file', pdf_path)
    except Exception as e:
        print(f"Something went wrong {e}")
pdf_path = "../data/raw_text/Gurbani_with_index_Uni.pdf"

## Step 2: Grab all the “raag lines” from the 1st page

In [2]:
raag_labels = get_page_text(pdf_path,page_number=0)
raag_labels

Text extracted from page 1!


'1\n❁❁❁❁❁❁❁❁❁❁❁❁❁❁ ❁❁❁❁❁❁❁❁❁❁❁❁❁❁\n❁\n❁\n❁\n❁ ੧ਓ ਸਤਿਗੁਰ ਪ੍ਰਸਾਤਿ ॥ ਰਾਗੁ ਟੋਡੀ ............................... ੭੧੧ ਰਾਗੁ ਪ੍ਰਭਾਿੀ ........................ ੧੩੨੭\n❁\n❁ ਰਾਗੁ ਿੈਰਾੜੀ ............................੭੧੯ ਰਾਗੁ ਜੈਜਾਵੰਿੀ ...................... ੧੩੫੨\nਿਿਕਰਾ ਰਾਗਾਂ ਕਾ\n❁\n❁ ਰਾਗੁ ਤਿਲੰਗ ............................ ੭੨੧ ਸਲੋਕ ਸਿਸਤਿਿੀ ਮਿਲਾ ੧ .... ੧੩੫੩\nਰਾਗ ........................................ ਪ੍ੰਨਾ ❁\n❁ ਰਾਗੁ ਸੂਿੀ .............................. ੭੨੮ ਸਲੋਕ ਸਿਸਤਿਿੀ ਮਿਲਾ ੫ .... ੧੩੫੩\nਜਪ੍ੁ ........................................... ੧\n❁\n❁ ਰਾਗੁ ਤਿਲਾਵਲੁ ........................ ੭੯੫ ਮਿਲਾ ੫ ਗਾਥਾ .....................੧੩੬੦\nਸੋ ਿਰੁ ........................................ ੮\n❁\n❁ ਰਾਗੁ ਗੋੋਂਡ .............................. ੮੫੯ ਫੁਨਿੇ ਮਿਲਾ ੫ ..................... ੧੩੬੧\nਸੋ ਪ੍ੁਰਖੁ ................................... ੧੦\n❁\n❁ ਰਾਗੁ ਰਾਮਕਲੀ ......................... ੮੭੬ ਚਉਿੋਲੇ ਮਿਲਾ ੫ .................. ੧੩੬੩\nਸੋਤਿਲਾ ................................... ੧੨\n❁\n❁ ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ ................. ੯੭੫ ਸਲੋਕ ਭਗਿ ਕਿੀਰ ਜੀਉ 

In [3]:
lines = raag_labels.splitlines(keepends=False)
lines

['1',
 '❁❁❁❁❁❁❁❁❁❁❁❁❁❁ ❁❁❁❁❁❁❁❁❁❁❁❁❁❁',
 '❁',
 '❁',
 '❁',
 '❁ ੧ਓ ਸਤਿਗੁਰ ਪ੍ਰਸਾਤਿ ॥ ਰਾਗੁ ਟੋਡੀ ............................... ੭੧੧ ਰਾਗੁ ਪ੍ਰਭਾਿੀ ........................ ੧੩੨੭',
 '❁',
 '❁ ਰਾਗੁ ਿੈਰਾੜੀ ............................੭੧੯ ਰਾਗੁ ਜੈਜਾਵੰਿੀ ...................... ੧੩੫੨',
 'ਿਿਕਰਾ ਰਾਗਾਂ ਕਾ',
 '❁',
 '❁ ਰਾਗੁ ਤਿਲੰਗ ............................ ੭੨੧ ਸਲੋਕ ਸਿਸਤਿਿੀ ਮਿਲਾ ੧ .... ੧੩੫੩',
 'ਰਾਗ ........................................ ਪ੍ੰਨਾ ❁',
 '❁ ਰਾਗੁ ਸੂਿੀ .............................. ੭੨੮ ਸਲੋਕ ਸਿਸਤਿਿੀ ਮਿਲਾ ੫ .... ੧੩੫੩',
 'ਜਪ੍ੁ ........................................... ੧',
 '❁',
 '❁ ਰਾਗੁ ਤਿਲਾਵਲੁ ........................ ੭੯੫ ਮਿਲਾ ੫ ਗਾਥਾ .....................੧੩੬੦',
 'ਸੋ ਿਰੁ ........................................ ੮',
 '❁',
 '❁ ਰਾਗੁ ਗੋੋਂਡ .............................. ੮੫੯ ਫੁਨਿੇ ਮਿਲਾ ੫ ..................... ੧੩੬੧',
 'ਸੋ ਪ੍ੁਰਖੁ ................................... ੧੦',
 '❁',
 '❁ ਰਾਗੁ ਰਾਮਕਲੀ ......................... ੮੭੬ ਚਉਿੋਲੇ ਮਿਲਾ ੫ .................. ੧੩੬੩',
 'ਸੋਤਿਲਾ .................................

In [4]:
for i, line in enumerate(lines):
    print(i,repr(line))

0 '1'
1 '❁❁❁❁❁❁❁❁❁❁❁❁❁❁ ❁❁❁❁❁❁❁❁❁❁❁❁❁❁'
2 '❁'
3 '❁'
4 '❁'
5 '❁ ੧ਓ ਸਤਿਗੁਰ ਪ੍ਰਸਾਤਿ ॥ ਰਾਗੁ ਟੋਡੀ ............................... ੭੧੧ ਰਾਗੁ ਪ੍ਰਭਾਿੀ ........................ ੧੩੨੭'
6 '❁'
7 '❁ ਰਾਗੁ ਿੈਰਾੜੀ ............................੭੧੯ ਰਾਗੁ ਜੈਜਾਵੰਿੀ ...................... ੧੩੫੨'
8 'ਿਿਕਰਾ ਰਾਗਾਂ ਕਾ'
9 '❁'
10 '❁ ਰਾਗੁ ਤਿਲੰਗ ............................ ੭੨੧ ਸਲੋਕ ਸਿਸਤਿਿੀ ਮਿਲਾ ੧ .... ੧੩੫੩'
11 'ਰਾਗ ........................................ ਪ੍ੰਨਾ ❁'
12 '❁ ਰਾਗੁ ਸੂਿੀ .............................. ੭੨੮ ਸਲੋਕ ਸਿਸਤਿਿੀ ਮਿਲਾ ੫ .... ੧੩੫੩'
13 'ਜਪ੍ੁ ........................................... ੧'
14 '❁'
15 '❁ ਰਾਗੁ ਤਿਲਾਵਲੁ ........................ ੭੯੫ ਮਿਲਾ ੫ ਗਾਥਾ .....................੧੩੬੦'
16 'ਸੋ ਿਰੁ ........................................ ੮'
17 '❁'
18 '❁ ਰਾਗੁ ਗੋੋਂਡ .............................. ੮੫੯ ਫੁਨਿੇ ਮਿਲਾ ੫ ..................... ੧੩੬੧'
19 'ਸੋ ਪ੍ੁਰਖੁ ................................... ੧੦'
20 '❁'
21 '❁ ਰਾਗੁ ਰਾਮਕਲੀ ......................... ੮੭੬ ਚਉਿੋਲੇ ਮਿਲਾ ੫ .................. ੧੩੬੩'
22 'ਸੋਤਿਲਾ ...................

### Step 3A: Filter lines that contain the word `ਰਾਗੁ`

In [5]:
# empty list to store only raag lines
raag_lines = [] 
for line in lines:
    if "ਰਾਗੁ" in line:
        print(repr(line))
        raag_lines.append(line)


'❁ ੧ਓ ਸਤਿਗੁਰ ਪ੍ਰਸਾਤਿ ॥ ਰਾਗੁ ਟੋਡੀ ............................... ੭੧੧ ਰਾਗੁ ਪ੍ਰਭਾਿੀ ........................ ੧੩੨੭'
'❁ ਰਾਗੁ ਿੈਰਾੜੀ ............................੭੧੯ ਰਾਗੁ ਜੈਜਾਵੰਿੀ ...................... ੧੩੫੨'
'❁ ਰਾਗੁ ਤਿਲੰਗ ............................ ੭੨੧ ਸਲੋਕ ਸਿਸਤਿਿੀ ਮਿਲਾ ੧ .... ੧੩੫੩'
'❁ ਰਾਗੁ ਸੂਿੀ .............................. ੭੨੮ ਸਲੋਕ ਸਿਸਤਿਿੀ ਮਿਲਾ ੫ .... ੧੩੫੩'
'❁ ਰਾਗੁ ਤਿਲਾਵਲੁ ........................ ੭੯੫ ਮਿਲਾ ੫ ਗਾਥਾ .....................੧੩੬੦'
'❁ ਰਾਗੁ ਗੋੋਂਡ .............................. ੮੫੯ ਫੁਨਿੇ ਮਿਲਾ ੫ ..................... ੧੩੬੧'
'❁ ਰਾਗੁ ਰਾਮਕਲੀ ......................... ੮੭੬ ਚਉਿੋਲੇ ਮਿਲਾ ੫ .................. ੧੩੬੩'
'❁ ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ ................. ੯੭੫ ਸਲੋਕ ਭਗਿ ਕਿੀਰ ਜੀਉ ਕੇ ..... ੧੩੬੪'
'ਤਸਰੀ ਰਾਗੁ ................................ ੧੪'
'❁ ਰਾਗੁ ਮਾਲੀ ਗਉੜਾ ................... ੯੮੪ ਸਲੋਕ ਸੇਖ ਫਰੀਿ ਕੇ .............. ੧੩੭੭'
'ਰਾਗੁ ਮਾਝ ................................. ੯੪'
'❁ ਰਾਗੁ ਮਾਰੂ .............................. ੯੮੯ ਸਵਯੇ ਸਰੀ ਮੁਖ ਿਾਕ ਮਿਲਾ ੫ ... ੧੩੮੫'
'ਰਾਗੁ ਗਉੜੀ ............................ ੧੫੧'
'❁ ਰਾਗ

In [6]:
len(raag_lines)

29

In [7]:
import re
raag_lines_copy = raag_lines
pattern = r"ਰਾਗੁ\s+[\u0A00-\u0A7F]+"

In [8]:
labels = re.findall(pattern, str(raag_lines_copy))

In [9]:
labels


['ਰਾਗੁ ਟੋਡੀ',
 'ਰਾਗੁ ਪ੍ਰਭਾਿੀ',
 'ਰਾਗੁ ਿੈਰਾੜੀ',
 'ਰਾਗੁ ਜੈਜਾਵੰਿੀ',
 'ਰਾਗੁ ਤਿਲੰਗ',
 'ਰਾਗੁ ਸੂਿੀ',
 'ਰਾਗੁ ਤਿਲਾਵਲੁ',
 'ਰਾਗੁ ਗੋੋਂਡ',
 'ਰਾਗੁ ਰਾਮਕਲੀ',
 'ਰਾਗੁ ਨਟ',
 'ਰਾਗੁ ਮਾਲੀ',
 'ਰਾਗੁ ਮਾਝ',
 'ਰਾਗੁ ਮਾਰੂ',
 'ਰਾਗੁ ਗਉੜੀ',
 'ਰਾਗੁ ਿੁਖਾਰੀ',
 'ਰਾਗੁ ਆਸਾ',
 'ਰਾਗੁ ਕੇਿਾਰਾ',
 'ਰਾਗੁ ਗੂਜਰੀ',
 'ਰਾਗੁ ਭੈਰਉ',
 'ਰਾਗੁ ਿੇਵਗੰਧਾਰੀ',
 'ਰਾਗੁ ਿਸੰਿੁ',
 'ਰਾਗੁ ਤਿਿਾਗੜਾ',
 'ਰਾਗੁ ਸਾਰਗ',
 'ਰਾਗੁ ਵਡਿੰਸੁ',
 'ਰਾਗੁ ਮਲਾਰ',
 'ਰਾਗੁ ਸੋਰਤਿ',
 'ਰਾਗੁ ਕਾਨੜਾ',
 'ਰਾਗੁ ਧਨਾਸਰੀ',
 'ਰਾਗੁ ਕਤਲਆਨ',
 'ਰਾਗੁ ਜੈਿਸਰੀ']

In [10]:
print(len(labels))
for i, name in enumerate(labels,start=1):
    print(i,repr(name))

30
1 'ਰਾਗੁ ਟੋਡੀ'
2 'ਰਾਗੁ ਪ੍ਰਭਾਿੀ'
3 'ਰਾਗੁ ਿੈਰਾੜੀ'
4 'ਰਾਗੁ ਜੈਜਾਵੰਿੀ'
5 'ਰਾਗੁ ਤਿਲੰਗ'
6 'ਰਾਗੁ ਸੂਿੀ'
7 'ਰਾਗੁ ਤਿਲਾਵਲੁ'
8 'ਰਾਗੁ ਗੋੋਂਡ'
9 'ਰਾਗੁ ਰਾਮਕਲੀ'
10 'ਰਾਗੁ ਨਟ'
11 'ਰਾਗੁ ਮਾਲੀ'
12 'ਰਾਗੁ ਮਾਝ'
13 'ਰਾਗੁ ਮਾਰੂ'
14 'ਰਾਗੁ ਗਉੜੀ'
15 'ਰਾਗੁ ਿੁਖਾਰੀ'
16 'ਰਾਗੁ ਆਸਾ'
17 'ਰਾਗੁ ਕੇਿਾਰਾ'
18 'ਰਾਗੁ ਗੂਜਰੀ'
19 'ਰਾਗੁ ਭੈਰਉ'
20 'ਰਾਗੁ ਿੇਵਗੰਧਾਰੀ'
21 'ਰਾਗੁ ਿਸੰਿੁ'
22 'ਰਾਗੁ ਤਿਿਾਗੜਾ'
23 'ਰਾਗੁ ਸਾਰਗ'
24 'ਰਾਗੁ ਵਡਿੰਸੁ'
25 'ਰਾਗੁ ਮਲਾਰ'
26 'ਰਾਗੁ ਸੋਰਤਿ'
27 'ਰਾਗੁ ਕਾਨੜਾ'
28 'ਰਾਗੁ ਧਨਾਸਰੀ'
29 'ਰਾਗੁ ਕਤਲਆਨ'
30 'ਰਾਗੁ ਜੈਿਸਰੀ'


In [11]:
unique_labels = sorted(set(labels))
unique_labels


['ਰਾਗੁ ਆਸਾ',
 'ਰਾਗੁ ਕਤਲਆਨ',
 'ਰਾਗੁ ਕਾਨੜਾ',
 'ਰਾਗੁ ਕੇਿਾਰਾ',
 'ਰਾਗੁ ਗਉੜੀ',
 'ਰਾਗੁ ਗੂਜਰੀ',
 'ਰਾਗੁ ਗੋੋਂਡ',
 'ਰਾਗੁ ਜੈਜਾਵੰਿੀ',
 'ਰਾਗੁ ਜੈਿਸਰੀ',
 'ਰਾਗੁ ਟੋਡੀ',
 'ਰਾਗੁ ਤਿਲਾਵਲੁ',
 'ਰਾਗੁ ਤਿਲੰਗ',
 'ਰਾਗੁ ਤਿਿਾਗੜਾ',
 'ਰਾਗੁ ਧਨਾਸਰੀ',
 'ਰਾਗੁ ਨਟ',
 'ਰਾਗੁ ਪ੍ਰਭਾਿੀ',
 'ਰਾਗੁ ਭੈਰਉ',
 'ਰਾਗੁ ਮਲਾਰ',
 'ਰਾਗੁ ਮਾਝ',
 'ਰਾਗੁ ਮਾਰੂ',
 'ਰਾਗੁ ਮਾਲੀ',
 'ਰਾਗੁ ਰਾਮਕਲੀ',
 'ਰਾਗੁ ਵਡਿੰਸੁ',
 'ਰਾਗੁ ਸਾਰਗ',
 'ਰਾਗੁ ਸੂਿੀ',
 'ਰਾਗੁ ਸੋਰਤਿ',
 'ਰਾਗੁ ਿਸੰਿੁ',
 'ਰਾਗੁ ਿੁਖਾਰੀ',
 'ਰਾਗੁ ਿੇਵਗੰਧਾਰੀ',
 'ਰਾਗੁ ਿੈਰਾੜੀ']

In [12]:
len(unique_labels)

30

In [13]:
for i, label in enumerate(unique_labels, start=1):
    print(i, repr(label))


1 'ਰਾਗੁ ਆਸਾ'
2 'ਰਾਗੁ ਕਤਲਆਨ'
3 'ਰਾਗੁ ਕਾਨੜਾ'
4 'ਰਾਗੁ ਕੇਿਾਰਾ'
5 'ਰਾਗੁ ਗਉੜੀ'
6 'ਰਾਗੁ ਗੂਜਰੀ'
7 'ਰਾਗੁ ਗੋੋਂਡ'
8 'ਰਾਗੁ ਜੈਜਾਵੰਿੀ'
9 'ਰਾਗੁ ਜੈਿਸਰੀ'
10 'ਰਾਗੁ ਟੋਡੀ'
11 'ਰਾਗੁ ਤਿਲਾਵਲੁ'
12 'ਰਾਗੁ ਤਿਲੰਗ'
13 'ਰਾਗੁ ਤਿਿਾਗੜਾ'
14 'ਰਾਗੁ ਧਨਾਸਰੀ'
15 'ਰਾਗੁ ਨਟ'
16 'ਰਾਗੁ ਪ੍ਰਭਾਿੀ'
17 'ਰਾਗੁ ਭੈਰਉ'
18 'ਰਾਗੁ ਮਲਾਰ'
19 'ਰਾਗੁ ਮਾਝ'
20 'ਰਾਗੁ ਮਾਰੂ'
21 'ਰਾਗੁ ਮਾਲੀ'
22 'ਰਾਗੁ ਰਾਮਕਲੀ'
23 'ਰਾਗੁ ਵਡਿੰਸੁ'
24 'ਰਾਗੁ ਸਾਰਗ'
25 'ਰਾਗੁ ਸੂਿੀ'
26 'ਰਾਗੁ ਸੋਰਤਿ'
27 'ਰਾਗੁ ਿਸੰਿੁ'
28 'ਰਾਗੁ ਿੁਖਾਰੀ'
29 'ਰਾਗੁ ਿੇਵਗੰਧਾਰੀ'
30 'ਰਾਗੁ ਿੈਰਾੜੀ'


## Step 5: Compare against the canonical 31 raags

In [14]:
known_raags = [
     "ਸਿਰੀ ਰਾਗੁ", "ਰਾਗੁ ਮਾਝ", "ਰਾਗੁ ਗਉੜੀ", "ਰਾਗੁ ਆਸਾ", "ਰਾਗੁ ਗੂਜਰੀ", "ਰਾਗੁ ਦੇਵਗੰਧਾਰੀ", "ਰਾਗੁ ਬਿਹਾਗੜਾ", 
    "ਰਾਗੁ ਵਡਹੰਸ", "ਰਾਗੁ ਸੋਰਠਿ", "ਰਾਗੁ ਧਨਾਸਰੀ", "ਰਾਗੁ ਜੈਤਸਰੀ", "ਰਾਗੁ ਟੋਡੀ", "ਰਾਗੁ ਬੈਰਾੜੀ", "ਰਾਗੁ ਤਿਲੰਗ", 
    "ਰਾਗੁ ਸੂਹੀ", "ਰਾਗੁ ਬਿਲਾਵਲ", "ਰਾਗੁ ਗੋਂਡ", "ਰਾਗੁ ਰਾਮਕਲੀ", "ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ", "ਰਾਗੁ ਮਾਲੀ ਗਉੜਾ", "ਰਾਗੁ ਮਾਰੂ", 
    "ਰਾਗੁ ਤੁਖਾਰੀ", "ਰਾਗੁ ਕੇਦਾਰਾ", "ਰਾਗੁ ਭੈਰਉ", "ਰਾਗੁ ਬਸੰਤ", "ਰਾਗੁ ਸਾਰਗ", "ਰਾਗੁ ਮਲਾਰ", "ਰਾਗੁ ਕਾਨੜਾ", 
    "ਰਾਗੁ ਕਲਿਆਨ", "ਰਾਗੁ ਪ੍ਰਭਾਤੀ", "ਰਾਗੁ ਜੈਜਾਵੰਤੀ"
]

extracted_raags = unique_labels

In [15]:
# find the missing raag
missing = set(known_raags) - set(extracted_raags)
missing

{'ਰਾਗੁ ਕਲਿਆਨ',
 'ਰਾਗੁ ਕੇਦਾਰਾ',
 'ਰਾਗੁ ਗੋਂਡ',
 'ਰਾਗੁ ਜੈਜਾਵੰਤੀ',
 'ਰਾਗੁ ਜੈਤਸਰੀ',
 'ਰਾਗੁ ਤੁਖਾਰੀ',
 'ਰਾਗੁ ਦੇਵਗੰਧਾਰੀ',
 'ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ',
 'ਰਾਗੁ ਪ੍ਰਭਾਤੀ',
 'ਰਾਗੁ ਬਸੰਤ',
 'ਰਾਗੁ ਬਿਲਾਵਲ',
 'ਰਾਗੁ ਬਿਹਾਗੜਾ',
 'ਰਾਗੁ ਬੈਰਾੜੀ',
 'ਰਾਗੁ ਮਾਲੀ ਗਉੜਾ',
 'ਰਾਗੁ ਵਡਹੰਸ',
 'ਰਾਗੁ ਸੂਹੀ',
 'ਰਾਗੁ ਸੋਰਠਿ',
 'ਸਿਰੀ ਰਾਗੁ'}

In [16]:
len(missing) # due to spelling issues and missing one raag

18

In [17]:
extra = set(extracted_raags) - set(known_raags)
extra

{'ਰਾਗੁ ਕਤਲਆਨ',
 'ਰਾਗੁ ਕੇਿਾਰਾ',
 'ਰਾਗੁ ਗੋੋਂਡ',
 'ਰਾਗੁ ਜੈਜਾਵੰਿੀ',
 'ਰਾਗੁ ਜੈਿਸਰੀ',
 'ਰਾਗੁ ਤਿਲਾਵਲੁ',
 'ਰਾਗੁ ਤਿਿਾਗੜਾ',
 'ਰਾਗੁ ਨਟ',
 'ਰਾਗੁ ਪ੍ਰਭਾਿੀ',
 'ਰਾਗੁ ਮਾਲੀ',
 'ਰਾਗੁ ਵਡਿੰਸੁ',
 'ਰਾਗੁ ਸੂਿੀ',
 'ਰਾਗੁ ਸੋਰਤਿ',
 'ਰਾਗੁ ਿਸੰਿੁ',
 'ਰਾਗੁ ਿੁਖਾਰੀ',
 'ਰਾਗੁ ਿੇਵਗੰਧਾਰੀ',
 'ਰਾਗੁ ਿੈਰਾੜੀ'}

In [18]:
len(extra)

17

In [19]:
len(known_raags)

31

### Offical Label List for Raag Labels

In [20]:
RAAG_MASTER = known_raags
print(len(RAAG_MASTER))
for i, raag in enumerate(RAAG_MASTER, start=1):
    print(i, repr(raag))

31
1 'ਸਿਰੀ ਰਾਗੁ'
2 'ਰਾਗੁ ਮਾਝ'
3 'ਰਾਗੁ ਗਉੜੀ'
4 'ਰਾਗੁ ਆਸਾ'
5 'ਰਾਗੁ ਗੂਜਰੀ'
6 'ਰਾਗੁ ਦੇਵਗੰਧਾਰੀ'
7 'ਰਾਗੁ ਬਿਹਾਗੜਾ'
8 'ਰਾਗੁ ਵਡਹੰਸ'
9 'ਰਾਗੁ ਸੋਰਠਿ'
10 'ਰਾਗੁ ਧਨਾਸਰੀ'
11 'ਰਾਗੁ ਜੈਤਸਰੀ'
12 'ਰਾਗੁ ਟੋਡੀ'
13 'ਰਾਗੁ ਬੈਰਾੜੀ'
14 'ਰਾਗੁ ਤਿਲੰਗ'
15 'ਰਾਗੁ ਸੂਹੀ'
16 'ਰਾਗੁ ਬਿਲਾਵਲ'
17 'ਰਾਗੁ ਗੋਂਡ'
18 'ਰਾਗੁ ਰਾਮਕਲੀ'
19 'ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ'
20 'ਰਾਗੁ ਮਾਲੀ ਗਉੜਾ'
21 'ਰਾਗੁ ਮਾਰੂ'
22 'ਰਾਗੁ ਤੁਖਾਰੀ'
23 'ਰਾਗੁ ਕੇਦਾਰਾ'
24 'ਰਾਗੁ ਭੈਰਉ'
25 'ਰਾਗੁ ਬਸੰਤ'
26 'ਰਾਗੁ ਸਾਰਗ'
27 'ਰਾਗੁ ਮਲਾਰ'
28 'ਰਾਗੁ ਕਾਨੜਾ'
29 'ਰਾਗੁ ਕਲਿਆਨ'
30 'ਰਾਗੁ ਪ੍ਰਭਾਤੀ'
31 'ਰਾਗੁ ਜੈਜਾਵੰਤੀ'


## Step 6: Get the entire PDF text into one big string
so far we only extracted page 0, Now we want _all the pages_.

In [21]:
def get_full_text(pdf_path):
    full_gurbani_text = ""
        
    with pdfplumber.open(pdf_path) as pdf:
        print(f'Total pages: {len(pdf.pages)}')
        for idx, page in enumerate(pdf.pages):
            full_text = page.extract_text()
            if full_text:
                full_gurbani_text += full_text + "\n\n"
            else:
                print(f"No text on page {idx}")
            return full_text
    return full_gurbani_text
pdf_path = "../data/raw_text/Gurbani_with_index_Uni.pdf"

In [22]:
FULL_GURBANI_TEXT = get_full_text(pdf_path)

Total pages: 1483


### Detect raag headings in `full_gurbani_text`
most raag headings look something like:
* `ਰਾਗੁ ਸਿਰੀ ਮਹਲਾ ੧`
* `ਰਾਗੁ ਗੂਜਰੀ ਮਹਲਾ ੫`

In [40]:
import re
pattern_heading = r"(ਰਾਗੁ\s+[\u0A00-\u0A7F]+|[\u0A00-\u0A7F]+\s+ਰਾਗੁ)"

raag_matches = list(re.finditer(pattern_heading,FULL_GURBANI_TEXT))
len(raag_matches)

31

In [24]:
for match in raag_matches:
    print(repr(match.group()))

'ਰਾਗੁ ਟੋਡੀ'
'੭੧੧ ਰਾਗੁ'
'ਰਾਗੁ ਿੈਰਾੜੀ'
'੭੧੯ ਰਾਗੁ'
'ਰਾਗੁ ਤਿਲੰਗ'
'ਰਾਗੁ ਸੂਿੀ'
'ਰਾਗੁ ਤਿਲਾਵਲੁ'
'ਰਾਗੁ ਗੋੋਂਡ'
'ਰਾਗੁ ਰਾਮਕਲੀ'
'ਰਾਗੁ ਨਟ'
'ਤਸਰੀ ਰਾਗੁ'
'ਰਾਗੁ ਮਾਲੀ'
'੧੩੭੭\nਰਾਗੁ'
'ਰਾਗੁ ਮਾਰੂ'
'੧੩੮੫\nਰਾਗੁ'
'ਰਾਗੁ ਿੁਖਾਰੀ'
'੧੪੧੦\nਰਾਗੁ'
'ਰਾਗੁ ਕੇਿਾਰਾ'
'੧੪੨੬\nਰਾਗੁ'
'ਰਾਗੁ ਭੈਰਉ'
'੧੪੨੯\nਰਾਗੁ'
'ਰਾਗੁ ਿਸੰਿੁ'
'੧੪੨੯\nਰਾਗੁ'
'ਰਾਗੁ ਸਾਰਗ'
'ਰਾਗੁ ਵਡਿੰਸੁ'
'ਰਾਗੁ ਮਲਾਰ'
'ਰਾਗੁ ਸੋਰਤਿ'
'ਰਾਗੁ ਕਾਨੜਾ'
'ਰਾਗੁ ਧਨਾਸਰੀ'
'ਰਾਗੁ ਕਤਲਆਨ'
'ਰਾਗੁ ਜੈਿਸਰੀ'


### Store name + position for each raag heading
Each `match` object has:
* `match.group()`
* `match.start()`

In [25]:
raag_positions = []

for match in raag_matches:
    raag_name = match.group()
    start_idx = match.start()
    raag_positions.append((raag_name, start_idx))


In [26]:
len(raag_positions)

31

In [27]:
raw_raag_names = sorted({name for name, _ in raag_positions})

print(len(raw_raag_names))
for i, r in enumerate(raw_raag_names, start=1):
    print(i, repr(r))

30
1 'ਤਸਰੀ ਰਾਗੁ'
2 'ਰਾਗੁ ਕਤਲਆਨ'
3 'ਰਾਗੁ ਕਾਨੜਾ'
4 'ਰਾਗੁ ਕੇਿਾਰਾ'
5 'ਰਾਗੁ ਗੋੋਂਡ'
6 'ਰਾਗੁ ਜੈਿਸਰੀ'
7 'ਰਾਗੁ ਟੋਡੀ'
8 'ਰਾਗੁ ਤਿਲਾਵਲੁ'
9 'ਰਾਗੁ ਤਿਲੰਗ'
10 'ਰਾਗੁ ਧਨਾਸਰੀ'
11 'ਰਾਗੁ ਨਟ'
12 'ਰਾਗੁ ਭੈਰਉ'
13 'ਰਾਗੁ ਮਲਾਰ'
14 'ਰਾਗੁ ਮਾਰੂ'
15 'ਰਾਗੁ ਮਾਲੀ'
16 'ਰਾਗੁ ਰਾਮਕਲੀ'
17 'ਰਾਗੁ ਵਡਿੰਸੁ'
18 'ਰਾਗੁ ਸਾਰਗ'
19 'ਰਾਗੁ ਸੂਿੀ'
20 'ਰਾਗੁ ਸੋਰਤਿ'
21 'ਰਾਗੁ ਿਸੰਿੁ'
22 'ਰਾਗੁ ਿੁਖਾਰੀ'
23 'ਰਾਗੁ ਿੈਰਾੜੀ'
24 '੧੩੭੭\nਰਾਗੁ'
25 '੧੩੮੫\nਰਾਗੁ'
26 '੧੪੧੦\nਰਾਗੁ'
27 '੧੪੨੬\nਰਾਗੁ'
28 '੧੪੨੯\nਰਾਗੁ'
29 '੭੧੧ ਰਾਗੁ'
30 '੭੧੯ ਰਾਗੁ'


In [28]:
# Build a filtered list that removes any entry that starts with a Gurmukhi digit

gurmukhi_digits = "੦੧੨੩੪੫੬੭੮੯"

valid_raw_raag_names = [
    r for r in raw_raag_names
    if r[0] not in gurmukhi_digits
]

print(len(valid_raw_raag_names))
for i, r in enumerate(valid_raw_raag_names, start=1):
    print(i, repr(r))

23
1 'ਤਸਰੀ ਰਾਗੁ'
2 'ਰਾਗੁ ਕਤਲਆਨ'
3 'ਰਾਗੁ ਕਾਨੜਾ'
4 'ਰਾਗੁ ਕੇਿਾਰਾ'
5 'ਰਾਗੁ ਗੋੋਂਡ'
6 'ਰਾਗੁ ਜੈਿਸਰੀ'
7 'ਰਾਗੁ ਟੋਡੀ'
8 'ਰਾਗੁ ਤਿਲਾਵਲੁ'
9 'ਰਾਗੁ ਤਿਲੰਗ'
10 'ਰਾਗੁ ਧਨਾਸਰੀ'
11 'ਰਾਗੁ ਨਟ'
12 'ਰਾਗੁ ਭੈਰਉ'
13 'ਰਾਗੁ ਮਲਾਰ'
14 'ਰਾਗੁ ਮਾਰੂ'
15 'ਰਾਗੁ ਮਾਲੀ'
16 'ਰਾਗੁ ਰਾਮਕਲੀ'
17 'ਰਾਗੁ ਵਡਿੰਸੁ'
18 'ਰਾਗੁ ਸਾਰਗ'
19 'ਰਾਗੁ ਸੂਿੀ'
20 'ਰਾਗੁ ਸੋਰਤਿ'
21 'ਰਾਗੁ ਿਸੰਿੁ'
22 'ਰਾਗੁ ਿੁਖਾਰੀ'
23 'ਰਾਗੁ ਿੈਰਾੜੀ'


In [29]:
gurmukhi_digits = "੦੧੨੩੪੫੬੭੮੯"

filtered_raag_positions = [(name,idx) for name, idx in raag_positions if name and name[0] not in gurmukhi_digits]
len(filtered_raag_positions)

23

### Build a mapping from raw-clean

In [30]:
RAAG_MAP = {
'ਰਾਗੁ ਟੋਡੀ': 'ਰਾਗੁ ਟੋਡੀ',
'ਰਾਗੁ ਪ੍ਰਭਾਿੀ': 'ਰਾਗੁ ਪ੍ਰਭਾਤੀ',
'ਰਾਗੁ ਿੈਰਾੜੀ': 'ਰਾਗੁ ਬੈਰਾੜੀ',
'ਰਾਗੁ ਜੈਜਾਵੰਿੀ': 'ਰਾਗੁ ਜੈਜਾਵੰਤੀ',
'ਰਾਗੁ ਤਿਲੰਗ': 'ਰਾਗੁ ਤਿਲੰਗ',
'ਰਾਗੁ ਸੂਿੀ': 'ਰਾਗੁ ਸੂਹੀ',
'ਰਾਗੁ ਤਿਲਾਵਲੁ': 'ਰਾਗੁ ਬਿਲਾਵਲ',
'ਰਾਗੁ ਗੋੋਂਡ': 'ਰਾਗੁ ਗੋਂਡ',
'ਰਾਗੁ ਰਾਮਕਲੀ': 'ਰਾਗੁ ਰਾਮਕਲੀ',
'ਰਾਗੁ ਨਟ': 'ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ',
'ਰਾਗੁ ਮਾਲੀ': 'ਰਾਗੁ ਮਾਲੀ ਗਉੜਾ',
'ਰਾਗੁ ਮਾਝ': 'ਰਾਗੁ ਮਾਝ',
'ਰਾਗੁ ਮਾਰੂ': 'ਰਾਗੁ ਮਾਰੂ',
'ਰਾਗੁ ਗਉੜੀ': 'ਰਾਗੁ ਗਉੜੀ',
'ਰਾਗੁ ਿੁਖਾਰੀ': 'ਰਾਗੁ ਤੁਖਾਰੀ',
'ਰਾਗੁ ਆਸਾ': 'ਰਾਗੁ ਆਸਾ',
'ਰਾਗੁ ਕੇਿਾਰਾ': 'ਰਾਗੁ ਕੇਦਾਰਾ',
'ਰਾਗੁ ਗੂਜਰੀ': 'ਰਾਗੁ ਗੂਜਰੀ',
'ਰਾਗੁ ਭੈਰਉ': 'ਰਾਗੁ ਭੈਰਉ',
'ਰਾਗੁ ਿੇਵਗੰਧਾਰੀ': 'ਰਾਗੁ ਦੇਵਗੰਧਾਰੀ',
'ਰਾਗੁ ਿਸੰਿੁ': 'ਰਾਗੁ ਬਸੰਤੁ',
'ਰਾਗੁ ਤਿਿਾਗੜਾ': 'ਰਾਗੁ ਬਿਹਾਗੜਾ',
'ਰਾਗੁ ਸਾਰਗ': 'ਰਾਗੁ ਸਾਰਗ',
'ਰਾਗੁ ਵਡਿੰਸੁ': 'ਰਾਗੁ ਵਡਹੰਸ',
'ਰਾਗੁ ਮਲਾਰ': 'ਰਾਗੁ ਮਲਾਰ',
'ਰਾਗੁ ਸੋਰਤਿ': 'ਰਾਗੁ ਸੋਰਠਿ',
'ਰਾਗੁ ਕਾਨੜਾ': 'ਰਾਗੁ ਕਾਨੜਾ',
'ਰਾਗੁ ਧਨਾਸਰੀ': 'ਰਾਗੁ ਧਨਾਸਰੀ',
'ਰਾਗੁ ਕਤਲਆਨ': 'ਰਾਗੁ ਕਲਿਆਨ',
'ਰਾਗੁ ਜੈਿਸਰੀ': 'ਰਾਗੁ ਜੈਤਸਰੀ',
'ਤਸਰੀ ਰਾਗੁ': 'ਸਿਰੀ ਰਾਗੁ'
}




###  Apply the mapping to `raag_positions`

In [31]:
canonical_positions = []

for raw_name, idx in raag_positions:
    canon_name = RAAG_MAP.get(raw_name, raw_name)
    canonical_positions.append((canon_name, idx))
canonical_positions

[('ਰਾਗੁ ਟੋਡੀ', 60),
 ('੭੧੧ ਰਾਗੁ', 102),
 ('ਰਾਗੁ ਬੈਰਾੜੀ', 153),
 ('੭੧੯ ਰਾਗੁ', 193),
 ('ਰਾਗੁ ਤਿਲੰਗ', 258),
 ('ਰਾਗੁ ਸੂਹੀ', 387),
 ('ਰਾਗੁ ਬਿਲਾਵਲ', 517),
 ('ਰਾਗੁ ਗੋਂਡ', 651),
 ('ਰਾਗੁ ਰਾਮਕਲੀ', 790),
 ('ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ', 920),
 ('ਸਿਰੀ ਰਾਗੁ', 989),
 ('ਰਾਗੁ ਮਾਲੀ ਗਉੜਾ', 1039),
 ('੧੩੭੭\nਰਾਗੁ', 1110),
 ('ਰਾਗੁ ਮਾਰੂ', 1165),
 ('੧੩੮੫\nਰਾਗੁ', 1238),
 ('ਰਾਗੁ ਤੁਖਾਰੀ', 1290),
 ('੧੪੧੦\nਰਾਗੁ', 1365),
 ('ਰਾਗੁ ਕੇਦਾਰਾ', 1416),
 ('੧੪੨੬\nਰਾਗੁ', 1491),
 ('ਰਾਗੁ ਭੈਰਉ', 1543),
 ('੧੪੨੯\nਰਾਗੁ', 1618),
 ('ਰਾਗੁ ਬਸੰਤੁ', 1669),
 ('੧੪੨੯\nਰਾਗੁ', 1748),
 ('ਰਾਗੁ ਸਾਰਗ', 1796),
 ('ਰਾਗੁ ਵਡਹੰਸ', 1839),
 ('ਰਾਗੁ ਮਲਾਰ', 1885),
 ('ਰਾਗੁ ਸੋਰਠਿ', 1928),
 ('ਰਾਗੁ ਕਾਨੜਾ', 1974),
 ('ਰਾਗੁ ਧਨਾਸਰੀ', 2017),
 ('ਰਾਗੁ ਕਲਿਆਨ', 2061),
 ('ਰਾਗੁ ਜੈਤਸਰੀ', 2101)]

In [32]:
len(canonical_positions)

31

In [33]:
gurmukhi_digits = "੦੧੨੩੪੫੬੭੮੯"

canonical_positions_clean = [(name,idx)for name,idx in canonical_positions if name and name[0] not in gurmukhi_digits]

In [34]:
len(canonical_positions_clean)

23

In [45]:
canonical_positions_clean

[('ਰਾਗੁ ਟੋਡੀ', 60),
 ('ਰਾਗੁ ਬੈਰਾੜੀ', 153),
 ('ਰਾਗੁ ਤਿਲੰਗ', 258),
 ('ਰਾਗੁ ਸੂਹੀ', 387),
 ('ਰਾਗੁ ਬਿਲਾਵਲ', 517),
 ('ਰਾਗੁ ਗੋਂਡ', 651),
 ('ਰਾਗੁ ਰਾਮਕਲੀ', 790),
 ('ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ', 920),
 ('ਸਿਰੀ ਰਾਗੁ', 989),
 ('ਰਾਗੁ ਮਾਲੀ ਗਉੜਾ', 1039),
 ('ਰਾਗੁ ਮਾਰੂ', 1165),
 ('ਰਾਗੁ ਤੁਖਾਰੀ', 1290),
 ('ਰਾਗੁ ਕੇਦਾਰਾ', 1416),
 ('ਰਾਗੁ ਭੈਰਉ', 1543),
 ('ਰਾਗੁ ਬਸੰਤੁ', 1669),
 ('ਰਾਗੁ ਸਾਰਗ', 1796),
 ('ਰਾਗੁ ਵਡਹੰਸ', 1839),
 ('ਰਾਗੁ ਮਲਾਰ', 1885),
 ('ਰਾਗੁ ਸੋਰਠਿ', 1928),
 ('ਰਾਗੁ ਕਾਨੜਾ', 1974),
 ('ਰਾਗੁ ਧਨਾਸਰੀ', 2017),
 ('ਰਾਗੁ ਕਲਿਆਨ', 2061),
 ('ਰਾਗੁ ਜੈਤਸਰੀ', 2101)]

In [43]:
FULL_GURBANI_TEXT.count("ਰਾਗੁ")


31

In [44]:
FULL_GURBANI_TEXT.count("ਰਾਗ")

34

In [47]:
len(canonical_positions_clean)

23